In [15]:
import pandas as pd

df = pd.read_csv('0.1_filtered_dataset.csv')

missing_cols = df.columns[df.isnull().any()].tolist()
print("MISSING COLUMNS BEFORE IMPUTATION")
print(missing_cols)
print()

stats = df[missing_cols].describe()

median_row = pd.DataFrame(df[missing_cols].median(), columns=['median']).T
stats = pd.concat([stats, median_row])

print(f"STATISTICS OF COLUMNS WITH MISSING DATA ({len(missing_cols)} columns)")
print(stats.round(2).to_string())
print()

for col in missing_cols:
    mean_val = round(df[col].mean(), 1)
    df[col] = df[col].fillna(mean_val)

remaining_missing = df.isnull().sum().sum()
print(f"Total missing cells after filling with mean: {remaining_missing}")

output_file = '0.2_processed_data.csv'
df.to_csv(output_file, index=False)
print(f"This dataset is ready for the next step'")

MISSING COLUMNS BEFORE IMPUTATION
['Age', 'Screen_Time_Hours', 'Sleep_Hours', 'Physical_Activity_Hours', 'Meditation_Minutes']

STATISTICS OF COLUMNS WITH MISSING DATA (5 columns)
             Age  Screen_Time_Hours  Sleep_Hours  Physical_Activity_Hours  Meditation_Minutes
count   49750.00           49150.00     49050.00                 48900.00            48800.00
mean       41.45               7.99         7.00                     4.99               30.03
std        13.87               3.47         1.73                     2.87               17.61
min        18.00               2.00         4.00                     0.00                0.00
25%        29.00               5.00         5.50                     2.50               15.00
50%        41.00               8.00         7.00                     5.00               30.00
75%        53.00              11.00         8.50                     7.40               45.00
max        65.00              14.00        10.00                    

In [16]:
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df_orig = pd.read_csv('0.1_filtered_dataset.csv')
df_proc = pd.read_csv('0.2_processed_data.csv')

output_dir = '/Users/hungdan/Documents/pj_02/early-burnout-risk-prediction/Assets'
os.makedirs(output_dir, exist_ok=True)

imputed_cols = df_orig.columns[df_orig.isnull().any()].tolist()
num_imputed = len(imputed_cols)

if num_imputed == 0:
    print("Không có giá trị thiếu nào trong tập dữ liệu gốc. Không cần vẽ biểu đồ so sánh.")
else:
    print("STATISTICAL COMPARISON (ORIGINAL VS PROCESSED)")
    for col in imputed_cols:
        orig_stats = df_orig[col].describe()
        proc_stats = df_proc[col].describe()
        
        comp_df = pd.DataFrame({
            'Original': orig_stats,
            'Processed': proc_stats,
            'Difference': proc_stats - orig_stats
        }).loc[['mean', 'std', 'min', '50%', 'max']]
        
        print(f"\n Feature: {col} ")
        print(comp_df.round(4).to_string())

    plot_ncols = 3
    plot_nrows = math.ceil(num_imputed / plot_ncols)
    
    fig, axes = plt.subplots(nrows=plot_nrows, ncols=plot_ncols, figsize=(18, 5 * plot_nrows))
    axes = axes.flatten() if num_imputed > 1 else [axes]
    sns.set_theme(style="whitegrid")
    
    for i, col in enumerate(imputed_cols):
        sns.kdeplot(df_orig[col].dropna(), color='#4C72B0', label='Original Data', ax=axes[i], linewidth=2)
        sns.kdeplot(df_proc[col], color='#DD8452', linestyle='--', label='Mean Imputed', ax=axes[i], linewidth=2)
        
        axes[i].set_title(f'Distribution Shift: {col}', fontsize=12, fontweight='bold')
        axes[i].set_ylabel('Density')
        axes[i].legend()
  
    for j in range(num_imputed, len(axes)):
        fig.delaxes(axes[j])
    
    plt.suptitle('Effect of Mean Imputation on Data Distributions', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    comp_img_name = '0.3_imputation_comparison_kde_processed.png'
    comp_img_path = os.path.join(output_dir, comp_img_name)
    plt.savefig(comp_img_path, dpi=300, bbox_inches='tight')
    print(f"\n![Distribution Comparison Plot]({comp_img_path})")
    plt.close()

STATISTICAL COMPARISON (ORIGINAL VS PROCESSED)

 Feature: Age 
      Original  Processed  Difference
mean   41.4482    41.4479     -0.0002
std    13.8716    13.8369     -0.0347
min    18.0000    18.0000      0.0000
50%    41.0000    41.4000      0.4000
max    65.0000    65.0000      0.0000

 Feature: Screen_Time_Hours 
      Original  Processed  Difference
mean    7.9913     7.9914      0.0001
std     3.4725     3.4429     -0.0296
min     2.0000     2.0000      0.0000
50%     8.0000     8.0000      0.0000
max    14.0000    14.0000      0.0000

 Feature: Sleep_Hours 
      Original  Processed  Difference
mean    6.9974     6.9975      0.0000
std     1.7288     1.7123     -0.0165
min     4.0000     4.0000      0.0000
50%     7.0000     7.0000      0.0000
max    10.0000    10.0000      0.0000

 Feature: Physical_Activity_Hours 
      Original  Processed  Difference
mean     4.988     4.9883      0.0003
std      2.870     2.8383     -0.0317
min      0.000     0.0000      0.0000
50%      5.

In [17]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('0.2_processed_data.csv')

print("DATA BEFORE ENCODING")
print(df.head(3))

ordinal_mapping = {
    'Burnout_Risk': {'Low': 0, 'Moderate': 1, 'High': 2},
    'Stress_Level': {'Low': 0, 'Moderate': 1, 'High': 2},
    'Sleep_Quality': {'Poor': 0, 'Average': 1, 'Good': 2, 'Excellent': 3}
}

for col, mapping in ordinal_mapping.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

le = LabelEncoder()
label_cols = ['Occupation', 'Education_Level', 'Gender', 'Employment_Status']

for col in label_cols:
    if col in df.columns:
        df[col] = le.fit_transform(df[col])
        mapping_dict = dict(zip(le.classes_, le.transform(le.classes_)))
        print(f"Label Mapping cho '{col}': {mapping_dict}")

if 'Chronic_Stress' in df.columns:
    df = pd.get_dummies(df, columns=['Chronic_Stress'], prefix='Chronic_OHE', dtype=int)

print("\n DATA AFTER ENCODING ")
print(df.info())

output_file = '0.3_scaling_data.csv'
df.to_csv(output_file, index=False)
print(f"\nDữ liệu đã được mã hóa thành công và lưu tại: '{output_file}'")

DATA BEFORE ENCODING
    Age  Gender Employment_Status  Work_Hours_Per_Week  Screen_Time_Hours  \
0  58.0    Male          Employed                   67                9.1   
1  24.0  Female           Student                   44               12.0   
2  59.0    Male          Employed                   60               10.2   

   Sleep_Hours Sleep_Quality  Physical_Activity_Hours  Meditation_Minutes  \
0          4.6          Poor                      8.9                 5.0   
1          4.7          Poor                      0.8                18.0   
2          6.3       Average                      6.9                30.0   

   Coffee_Cups_Per_Day Stress_Level Chronic_Stress Burnout_Risk  \
0                    3         High            Yes         High   
1                    3     Moderate             No     Moderate   
2                    3     Moderate             No     Moderate   

          Occupation Education_Level  
0  Marketing Manager         Diploma  
1         Free